# Hospital Readmission EDA: Diabetes 130-US Hospitals

This data covers hospital visits of diabetic patients from 130 US hospitals, 1999–2008.

I am exploring the dataset to find out which factors are linked to a patient coming back within 30 days.

This notebook covers cleaning the data and exploring it with charts.

Source: UCI Machine Learning Repository

In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)  # show all columns when previewing

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/manishasingh07/data-set/IDS_mapping.csv
/kaggle/input/datasets/manishasingh07/data-set/diabetic_data.csv
/kaggle/input/datasets/brandao/diabetes/description.pdf
/kaggle/input/datasets/brandao/diabetes/diabetic_data.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/brandao/diabetes/diabetic_data.csv')
ids = pd.read_csv('/kaggle/input/datasets/manishasingh07/data-set/IDS_mapping.csv')
ids

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
...,...,...
62,22,Transfer from hospital inpt/same fac reslt in...
63,23,Born inside this hospital
64,24,Born outside this hospital
65,25,Transfer from Ambulatory Surgery Center


In [3]:
df.shape

(101766, 50)

101,766 rows and 50 columns, from 130 US hospitals (1999–2008).
Each row is one hospital visit (encounter): there are 101,766 unique encounter_id values.
There are 71,518 unique patients, so some patients appear many times. One patient had 40 visits.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

diag_1                    101766 non-null  object valuse is in float .
max_glu_serum             5346 non-null    object - has the 5346 entry.
A1Cresult                 17018  has this entry.
Pyers code is object but the value ?

In [5]:
df.head()
df.loc[0]

encounter_id                                 2278392
patient_nbr                                  8222157
race                                       Caucasian
gender                                        Female
age                                           [0-10)
weight                                             ?
admission_type_id                                  6
discharge_disposition_id                          25
admission_source_id                                1
time_in_hospital                                   1
payer_code                                         ?
medical_specialty           Pediatrics-Endocrinology
num_lab_procedures                                41
num_procedures                                     0
num_medications                                    1
number_outpatient                                  0
number_emergency                                   0
number_inpatient                                   0
diag_1                                        

age is in in thformat [0-10)
weight is not presesnt.
time_in_hospital is should in minutes .
Payers code is not present.
Dia2_,dia 3 is not presesnt -' ?'
max_glu_serum,A1Cresult    are NAN

age as [0-10):, it's stored as text ranges, not numbers, so it's an ordered category, and you'll later decide whether to keep it that way or convert it.

weight and payer_code "not present": They are present as columns but filled with "?", which means unknown, and you'll soon measure how much of each column is like that.

time_in_hospital in minutes: This one needs a rethink; check the smallest and largest values in that column and ask yourself whether minutes makes sense for a hospital stay.

diag_2 and diag_3 = "?": Correct for this row; also notice these are diagnosis codes (ICD-9), so even though many look like numbers, they're labels, not quantities.

max_glu_serum and A1Cresult = NaN: This is the important one: the original file contains the word "None" here, meaning the test was not performed, and pandas automatically turned it into NaN, so this NaN is real information, not missing data, and you'll need to handle it differently from "?".

There are 50 columns.

In [6]:
df.loc[0]

encounter_id                                 2278392
patient_nbr                                  8222157
race                                       Caucasian
gender                                        Female
age                                           [0-10)
weight                                             ?
admission_type_id                                  6
discharge_disposition_id                          25
admission_source_id                                1
time_in_hospital                                   1
payer_code                                         ?
medical_specialty           Pediatrics-Endocrinology
num_lab_procedures                                41
num_procedures                                     0
num_medications                                    1
number_outpatient                                  0
number_emergency                                   0
number_inpatient                                   0
diag_1                                        

In [7]:
df ["encounter_id"].value_counts()

encounter_id
443867222    1
2278392      1
149190       1
64410        1
500364       1
            ..
28236        1
15738        1
12522        1
63768        1
55842        1
Name: count, Length: 101766, dtype: int64

: "Each row is one hospital encounter."

In [8]:
df ["patient_nbr"].value_counts()

patient_nbr
88785891     40
43140906     28
88227540     23
1660293      23
23199021     23
             ..
174477542     1
38726739      1
77391171      1
89869032      1
63555939      1
Name: count, Length: 71518, dtype: int64

In [9]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [10]:
df[["encounter_id","patient_nbr"]].head()

,encounter_id,patient_nbr
0,2278392,8222157
1,149190,55629189
2,64410,86047875
3,500364,82442376
4,16680,42519267


In [11]:
df["patient_nbr"].value_counts().head()

patient_nbr
88785891    40
43140906    28
88227540    23
1660293     23
23199021    23
Name: count, dtype: int64

"Most frequent patient: 40 encounters; repetition is heavy for a few patients."

In [12]:
filt =df["patient_nbr"]== 88785891
df.loc[filt ,["encounter_id","patient_nbr","readmitted"]].value_counts()

encounter_id  patient_nbr  readmitted
119039172     88785891     >30           1
125094312     88785891     <30           1
126171582     88785891     >30           1
137245596     88785891     <30           1
139425576     88785891     <30           1
141994242     88785891     <30           1
150986298     88785891     <30           1
151413846     88785891     <30           1
152188656     88785891     <30           1
153558456     88785891     <30           1
154527444     88785891     >30           1
157363182     88785891     <30           1
158500410     88785891     >30           1
168920160     88785891     >30           1
174689286     88785891     <30           1
175632402     88785891     >30           1
183194856     88785891     >30           1
191345028     88785891     >30           1
205106328     88785891     <30           1
210079272     88785891     >30           1
221588430     88785891     <30           1
223647786     88785891     <30           1
225428574     88

Dataset overview

101,766 rows and 50 columns, from 130 US hospitals (1999–2008).
Each row is one hospital visit (encounter): there are 101,766 unique encounter_id values.
There are 71,518 unique patients, so some patients appear many times. One patient had 40 visits.

Missing data

Missing values are written as "?", not left blank, so pandas does not detect them automatically.
max_glu_serum and A1Cresult show NaN when the test was not done. This is real information, not missing data.
Glucose test done in only [ ]% of visits; A1C test done in only [ ]%.

Column types

age is stored as ranges like [0-10), not exact numbers.
admission_type_id, discharge_disposition_id, and admission_source_id are numbers, but they are codes for categories.
diag_1, diag_2, and diag_3 are diagnosis codes (ICD-9), stored as text.

Target: readmitted

Three values: NO (54,864), >30 (35,545), <30 (11,357).
Readmitted within 30 days: only [ ]%, so the target is imbalanced.
The label comes from hospital records, so some return visits (for example, at other hospitals) are not in the data as rows.

In [13]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Some return visits are not in the dataset, so the readmitted label comes from hospital records, not from the rows.

In [14]:
percentage = df["readmitted"].value_counts(normalize=True)*100
percentage

readmitted
NO     53.911916
>30    34.928169
<30    11.159916
Name: proportion, dtype: float64

Readmitted within 30 days: only 11.16%."

In [15]:
filt =(df == "?").sum()
filt

encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum                   0
A1Cresult                       0
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

In [16]:
df.tail(5)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,MC,?,51,0,16,0,0,0,250.13,291,458,9,NaN,>8,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,MC,?,33,3,18,0,0,1,560,276,787,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,MC,?,53,0,9,1,0,0,38,590,296,13,NaN,NaN,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,MC,Surgery-General,45,2,21,0,0,1,996,285,998,9,NaN,NaN,No,No,No,No,No,No,Steady,No,No,Steady,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
101765,443867222,175429310,Caucasian,Male,[70-80),?,1,1,7,6,?,?,13,3,3,0,0,0,530,530,787,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO


In [17]:
df.sample(5)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
98471,401407760,41120514,Caucasian,Female,[70-80),?,3,1,1,6,MD,?,40,0,17,0,0,2,518,486,428,9,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
5661,29159382,25606224,Caucasian,Female,[70-80),?,5,6,17,9,?,?,25,5,26,1,0,0,575,572,576,8,>200,NaN,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,>30
59828,167944980,80846541,Caucasian,Male,[60-70),?,6,1,17,1,?,Cardiology,44,0,7,0,0,0,728,414,719,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
17998,65634210,24135570,Caucasian,Female,[50-60),?,5,6,17,5,?,Family/GeneralPractice,17,1,22,1,0,6,493,242,300,5,>200,NaN,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
33719,106335870,4068351,AfricanAmerican,Female,[70-80),?,2,1,1,10,MC,?,51,1,14,0,0,4,250.7,707,496,5,NaN,>8,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO


### What I noticed in the random rows

- **weight**, **payer_code**, and **medical_specialty** contain `?` in several rows, which means the value is unknown. I will measure how much of each column is missing in step 10.
- **number_inpatient** and **number_outpatient** show 0 or 1 in these rows. They are counts of visits in the year before this stay, so other rows may have higher values. I will check the full range with `describe()`.
- **max_glu_serum** and **A1Cresult** show `NaN`. Here this means the test was not done, so it is real information, not missing data.
- **insulin** shows "No" in these rows. The medicine columns can also contain other values such as "Steady", "Up", and "Down". I will check all values with `value_counts()`.

In [18]:
df.describe()

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.026403e+08,3.869636e+07,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.523890e+08,4.550514e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,2.302709e+08,8.754595e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,4.438672e+08,1.895026e+08,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


### Summary statistics (describe)

- time_in_hospital ranges from 1 to 14, so it is measured in **days**, not minutes.
- encounter_id, patient_nbr and the three *_id columns are identifiers or category codes, so their mean and std are not meaningful.
- Most patients had no outpatient, emergency or inpatient visits in the previous year (median = 0), but the maximums are [42], [76 ] and [ 21], so a small group of patients visit very often.
- On average a visit had about 43 lab procedures, 16 medications and 7 diagnoses.

In [19]:
df["insulin"].value_counts()

insulin
No        47383
Steady    30849
Down      12218
Up        11316
Name: count, dtype: int64

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

### Column types

pandas reads 13 columns as numbers (int64) and 37 as text (object). But not every number column is a real amount, so I sorted them by meaning:

**Numeric (8)**: real amounts that can be averaged
- time_in_hospital (days, 1–14)
- num_lab_procedures, num_procedures, num_medications
- number_outpatient, number_emergency, number_inpatient (visits in the previous year)
- number_diagnoses

**IDs (2)**: identify a row, not used for analysis
- encounter_id (one hospital visit)
- patient_nbr (one patient)

**Categorical stored as numbers (3)**: codes explained in IDS_mapping.csv
- admission_type_id, discharge_disposition_id, admission_source_id

**Categorical (37)**: text labels
- Patient info: race, gender, age (10-year ranges), weight
- Stay: payer_code, medical_specialty
- Diagnoses: diag_1, diag_2, diag_3 (ICD-9 codes, labels not numbers)
- Lab tests: max_glu_serum, A1Cresult
- 23 medication columns (metformin … metformin-pioglitazone), values: No / Steady / Up / Down
- change, diabetesMed
- Target: readmitted (NO, >30, <30)



In [21]:
df ["gender"].value_counts()

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

3 of the gender are unknown.

In [22]:
missing_pct = (df == "?").mean() * 100
missing_pct = missing_pct[missing_pct > 0]
missing_pct = missing_pct.sort_values(ascending=False)
missing_pct

weight               96.858479
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64

### Missing values ("?")

- weight has 96.9% missing values, so almost no patient has a recorded weight.
- medical_specialty has 49.1% missing values, so the doctor's specialty is unknown for about half the visits.
- payer_code has 39.6% missing values, so the payment type is unknown for about 4 in 10 visits.
- race has 2.2% missing values.
- diag_3 (1.4%), diag_2 (0.4%) and diag_1 (0.02%) have very few missing values.
- All other columns have no "?".

### Keep or drop decisions

| Column | Missing | Decision | Reason |
|---|---|---|---|
| weight | 96.9% | Drop column | Almost no patient has a recorded weight. |
| payer_code | 39.6% | Drop column | How a patient pays does not describe their health, and 40% is unknown. |
| medical_specialty | 49.1% | Keep, fill "?" with "Unknown" | The type of doctor (for example surgery vs. general medicine) may be linked to how sick patients are, and so to readmission. |
| race | 2.2% | Drop rows with "?" | Very few rows are missing, so removing them loses little data. |
| diag_1, diag_2, diag_3 | 0.02–1.4% | Keep, fill "?" with "Unknown" | Diagnoses can help explain readmission, and very few values are missing. |
| gender | 3 rows "Unknown/Invalid" | Keep | Only 3 rows, too few to affect results. |
| encounter_id | 0% | Keep for now, drop after step 14 | It is unique, so I use it to sort visits and find each patient's first visit. It does not explain readmission. |
| max_glu_serum, A1Cresult | NaN = test not done | Fill NaN with "None" | NaN here is real information, not missing data. |

In [23]:
df = df.drop(columns=["weight", "payer_code"])
df.shape

(101766, 48)

Droped the weight and payer_code

In [24]:
df = df[df["race"] != "?"]
df.shape

(99493, 48)

Drop the race column that had teh missing value ? 

In [25]:
cols = ["medical_specialty", "diag_1", "diag_2", "diag_3"]
df[cols] = df[cols].replace("?", "Unknown")

Replaced the ? missimg value to "Unknown"

In [26]:
df[["max_glu_serum", "A1Cresult"]] = df[["max_glu_serum", "A1Cresult"]].fillna("None")


Fill the lab test NaN with “None”

In [27]:
missing_pct = (df == "?").mean() * 100
missing_pct = missing_pct[missing_pct > 0]
missing_pct = missing_pct.sort_values(ascending=False)
missing_pct

Series([], dtype: float64)

In [28]:
df.shape

(99493, 48)